# Uber Fare Prediction: Linear Regression vs Random Forest

Place `uber.csv` in this folder. Expected columns include `fare_amount`, `pickup_longitude`, `pickup_latitude`, `dropoff_longitude`, `dropoff_latitude`, and `passenger_count`. The notebook cleans data, identifies outliers, checks correlations, trains both models, and compares R² and RMSE.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_csv("uber.csv")
df = df.drop(columns=[c for c in ["Unnamed: 0", "key"] if c in df.columns], errors="ignore")
need = ["fare_amount", "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "passenger_count"]
df = df.dropna(subset=[c for c in need if c in df.columns])
df = df[(df.fare_amount > 0) & (df.fare_amount < 500)]
for c in ["pickup_longitude", "dropoff_longitude"]:
    df = df[df[c].between(-180, 180)]
for c in ["pickup_latitude", "dropoff_latitude"]:
    df = df[df[c].between(-90, 90)]
df = df[df.passenger_count.between(1, 6)]
print("Clean shape:", df.shape)
df.head()

### Outlier identification (IQR method)

In [ ]:
q1, q3 = df.fare_amount.quantile([.25, .75])
iqr = q3 - q1
outliers = df[(df.fare_amount < q1 - 1.5*iqr) | (df.fare_amount > q3 + 1.5*iqr)]
print("Fare outliers:", len(outliers))
df = df[df.fare_amount.between(max(0, q1-1.5*iqr), q3+1.5*iqr)]

### Correlation

In [ ]:
numeric = df.select_dtypes(include="number")
print(numeric.corr().round(2))
numeric.corr().style.background_gradient(cmap="coolwarm")

### Train and compare models

In [ ]:
features = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "passenger_count"]
X, y = df[features], df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
models = {
    "Linear Regression": make_pipeline(SimpleImputer(), StandardScaler(), LinearRegression()),
    "Random Forest": make_pipeline(SimpleImputer(), RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({"Model": name, "R2": r2_score(y_test, pred),
                    "RMSE": mean_squared_error(y_test, pred)**.5})
print(pd.DataFrame(results).round(3).to_string(index=False))